# SkinIQ — multi-label skin concern model

EfficientNet-V2-S with **six independent sigmoid outputs** (acne, blackheads, hyperpigmentation,
redness, enlarged pores, wrinkles), trained with binary cross-entropy, so one face can have
several concerns, or none.

**Data:** your two Kaggle datasets, the Roboflow *Skin Problem Detection* dataset (boxes converted to
six image-level labels), and your annotations. Images are resized to 224 × 224.

**Before you start:** label at least the 300 test + 200 validation photos with
`annotate_colab.ipynb` (more training labels give a better model).

**Run:** Runtime → T4 GPU, then Runtime → Run all (about 45–75 min). Results are saved to
`My Drive/SkinIQ/models/skin_concerns/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/SkinIQ"
SKIN_TYPE_ZIP = "Skin Type Dataset - olimov.zip"
SKIN_CONDITION_ZIP = "Skin Issues Dataset - drishya.zip"
SKIN_PROBLEM_ZIP = "Skin Problem Detection.zip"   # Roboflow, CC BY 4.0
EPOCHS = 20
BATCH_SIZE = 32      # lower to 16 if you get "CUDA out of memory"

## 1. Set up

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
from google.colab import drive
drive.mount("/content/drive")
LABELS = f"{DRIVE_DIR}/labels/multilabel_labels.csv"
for f in (SKIN_TYPE_ZIP, SKIN_CONDITION_ZIP, SKIN_PROBLEM_ZIP):
    assert os.path.exists(f"{DRIVE_DIR}/{f}"), f"Not found: {DRIVE_DIR}/{f}"
assert os.path.exists(LABELS), "No labels yet: run annotate_colab.ipynb first"

%cd /content
!rm -rf SkinIQ && git clone -q https://github.com/rwiie/SkinIQ.git
%cd /content/SkinIQ
!pip install -q "timm>=1.0" opencv-python-headless
!mkdir -p data/raw
!cp "{DRIVE_DIR}/{SKIN_TYPE_ZIP}" "{DRIVE_DIR}/{SKIN_CONDITION_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_TYPE_ZIP}" -x "__MACOSX/*" -d data/raw
!unzip -q -o "/tmp/{SKIN_CONDITION_ZIP}" -x "__MACOSX/*" -d data/raw
!rm -rf data/raw/skin_type data/raw/skin_condition
!mv "data/raw/Skin Type Dataset - olimov" data/raw/skin_type
!mv "data/raw/Skin Issues Dataset - drishya" data/raw/skin_condition
!python training/prepare_data.py
!rm -rf data/raw/skin_problem && mkdir -p data/raw/skin_problem
!cp "{DRIVE_DIR}/{SKIN_PROBLEM_ZIP}" /tmp/
!unzip -q -o "/tmp/{SKIN_PROBLEM_ZIP}" -d data/raw/skin_problem
!python training/prepare_roboflow.py
!cp "{LABELS}" data/labels/multilabel_labels.csv

## 2. Label summary

In [ ]:
import pandas as pd
labels = pd.read_csv("data/labels/multilabel_labels.csv")
usable = labels[labels.unusable == 0]
concerns = ["acne", "blackheads", "hyperpigmentation", "redness", "enlarged_pores", "wrinkles"]
summary = usable.groupby("split")[concerns].sum()
summary["clear"] = usable[usable[concerns].sum(axis=1) == 0].groupby("split").size()
summary["2+ concerns"] = usable[usable[concerns].sum(axis=1) >= 2].groupby("split").size()
summary["photos"] = usable.groupby("split").size()
display(summary.fillna(0).astype(int))
n_test, n_val = (usable.split == "test").sum(), (usable.split == "val").sum()
print(f"test {n_test} / 300, val {n_val} / 200 labelled")
if n_test < 300 or n_val < 200:
    print("⚠️  Fewer than the recommended test/val labels: results will be less reliable.")

## 3. Train and evaluate (per concern)

In [ ]:
OUT = f"{DRIVE_DIR}/models"
!python training/train_multilabel.py --epochs {EPOCHS} --batch-size {BATCH_SIZE} --out-dir "{OUT}"

## 4. Whole image vs face regions (YuNet)

Scores each labelled test photo as a whole image and by face regions, and compares them.

In [ ]:
!python training/evaluate_regions.py --model "{OUT}/skin_concerns/model.pt"

## 5. Results

In [ ]:
import json, matplotlib.pyplot as plt
d = f"{DRIVE_DIR}/models/skin_concerns"
m = json.load(open(f"{d}/metrics.json"))
print("Test (photo level):", m["test"])
per = pd.read_csv(f"{d}/per_concern.csv").set_index("concern")
display(per)
ax = per[["precision", "recall", "f1", "roc_auc"]].plot.bar(figsize=(9, 3.5), ylim=(0, 1), rot=0,
                                                         title="Multi-label test results per concern")
plt.tight_layout(); plt.savefig(f"{d}/per_concern.png", dpi=150); plt.show()
h = pd.read_csv(f"{d}/history.csv")
h.plot(x="epoch", y=[c for c in h.columns if c.startswith(("train_loss", "val_"))], figsize=(9, 3), title="Training")
plt.tight_layout(); plt.savefig(f"{d}/training_curves.png", dpi=150); plt.show()
display(pd.read_csv(f"{d}/region_evaluation.csv"))